###DataLake (Deltalake) + Lakehouse (Deltatables) - using Delta format (parquet+snappy+delta log)

Delta Lake is an open-source storage framework that brings reliability, ACID transactions, and performance to data lakes. It sits on top of Parquet files and is most commonly used with Apache Spark and Databricks.<br>
Delta Lake & Deltalakhouse is the Core/Analytical storage layer behind Bronze–Silver–Gold (medallion) architectures.
<img src="https://pages.databricks.com/rs/094-YMS-629/images/delta-lake-logo-whitebackground.png" style="width:300px; float: right"/>

## ![](https://pages.databricks.com/rs/094-YMS-629/images/delta-lake-tiny-logo.png) Creating our first Delta Lake table

Delta is the default file and table format using Databricks.

Delta support parquet data only 

iceberg support parquet / orc/ avro

![](https://docs.databricks.com/aws/en/assets/images/well-architected-lakehouse-7d7b521addc268ac8b3d597bafa8cae9.png)

In [0]:
%sql
SELECT current_schema()

In [0]:
%sql
/*drop table lakehousecat1.deltadb.customer_txn;
drop table lakehousecat1.deltadb.customer_txn_part;
drop table lakehousecat1.deltadb.drugstbl;
drop table lakehousecat1.deltadb.drugstbl_merge;
drop table lakehousecat1.deltadb.drugstbl_partitioned;
drop table lakehousecat1.deltadb.employee_dv_demo1;
drop table lakehousecat1.deltadb.product_inventory;
drop table lakehousecat1.deltadb.tblsales;*/

In [0]:
#spark.sql(f"drop catalog if exists lakehousecat1 cascade")
spark.sql(f"CREATE CATALOG IF NOT EXISTS lakehousecat1")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS lakehousecat1.deltadb;")
spark.sql(f"""CREATE VOLUME IF NOT EXISTS lakehousecat1.deltadb.datalake;""")
#spark.sql(f"""CREATE VOLUME IF NOT EXISTS lakehousecat1.deltadb.deltavolume;""")
#spark.sql(f"""CREATE VOLUME IF NOT EXISTS lakehousecat1.deltadb.deltavolume2;""")



####1. Write data into delta file (Datalake) and table (Lakehouse)
1. How to migrate csv to delta format
2. Difference between Delta and Parquet
2. How to create Datalake & Lakehouse

In [0]:
#1. How to migrate csv to delta format (Delta Lake creation)
df = spark.read.csv('/Volumes/lakehousecat1/deltadb/datalake/druginfo.csv',header=True,inferSchema=True)#Reading normal data from datalake
df.write.format("delta").mode("overwrite").save("/Volumes/lakehousecat1/deltadb/datalake/targetdir")#writing normal data into deltalake(deltalake)
#2. Difference between Delta and Parquet
df.write.format("parquet").mode("overwrite").save("dbfs:/Volumes/lakehousecat1/deltadb/datalake/targetdirparquet")#writing normal data into parquet(datalake)
df.write.mode("overwrite").save("/Volumes/lakehousecat1/deltadb/datalake/targetdirdefaultdeltaparquet")#Databricks default format is delta(parquet)
#3. How to create Delta Lakehouse
spark.sql("drop table if exists lakehousecat1.deltadb.drugstbl")
df.write.saveAsTable("lakehousecat1.deltadb.drugstbl",mode='overwrite')#writing normal data from deltalakehouse(lakehouse)
#behind it stores the data in deltafile format in the s3 bucket (location is hidden for us in databricks free edition)

In [0]:
%sql
--under the hood data is stored in S3
explain select * from lakehousecat1.deltadb.drugstbl

#####We can have schema evolution performed

In [0]:
#We can add Schema evolution feature just by adding the below option in Delta tables.
#df.write.option("mergeSchema","True").saveAsTable("lakehousecat1.deltadb.drugstbl",mode='overwrite')

####2. DML Operations in Delta Tables & Files
- We are overcoming the WORM (Write Once Read Many) limitation in Cloud S3/GCS/ADLS or in Distributed storage layers like HDFS
- Delta file/table supports WMRM(Write Manay Read Many) operations, using DMLs such as  INSERT/DELETE/UPDATE/MERGE

In [0]:
%sql
--DDL is supportive (we will do more of these further)
create or replace table lakehousecat1.deltadb.sampletable(id int,name string) 
using delta;
insert into lakehousecat1.deltadb.sampletable values(1,'hduser');--Though the data is stored internally in delta file, we can't see the data in delta format in databricks serverless

insert into lakehousecat1.deltadb.sampletable values(2,'IZUSER');

UPDATE lakehousecat1.deltadb.sampletable SET name='WD37-user' WHERE id=2;

select * from lakehousecat1.deltadb.sampletable;
    
--DML - merge is possible in the delta tables/files
describe history lakehousecat1.deltadb.sampletable;

In [0]:
%sql
insert into lakehousecat1.deltadb.sampletable values(2,'IZUSER');
insert into lakehousecat1.deltadb.sampletable values(2,'IZUSER');
insert into lakehousecat1.deltadb.sampletable values(2,'IZUSER');
insert into lakehousecat1.deltadb.sampletable values(2,'IZUSER');
insert into lakehousecat1.deltadb.sampletable values(2,'IZUSER');


insert into lakehousecat1.deltadb.sampletable values(2,'IZUSER'),(2,'IZUSER'),(2,'IZUSER'),(2,'IZUSER'),(2,'IZUSER'),(2,'IZUSER'),(2,'IZUSER'),(2,'IZUSER'),(2,'IZUSER'),(2,'IZUSER');

-- every commit / successful operation creates a new version
-- every changes in the table creates a new version

-- how many version we can have it table (no limit) - no limit based on on version num  based on ts we have some soft limit 



-- checkpointing / optimize / vaccum 

In [0]:
%sql


-- version as of or timestamp 
select * from lakehousecat1.deltadb.sampletable version as of 10;

describe formatted lakehousecat1.deltadb.sampletable;

In [0]:
%sql
use lakehousecat1.deltadb

In [0]:
%sql
DESC HISTORY lakehousecat1.deltadb.drugstbl
-- df.saveAsTable

In [0]:
%sql
--DQL is supported
SELECT * FROM drugstbl where uniqueid=163740;

#####a. Table Update

In [0]:
%sql
--DML - update is possible in the delta tables/files
UPDATE drugstbl
  SET rating=rating-1
where uniqueid=163740;

In [0]:
%sql
--default latest version will be shown
SELECT * FROM drugstbl 
where uniqueid=163740;

SELECT * FROM drugstbl VERSION AS OF 0
where uniqueid=163740;

#####b. Table Delete

In [0]:
%sql
--DML - Delete is possible on delta tables/files
DELETE FROM drugstbl
where uniqueid=163740;

In [0]:
%sql
SELECT * FROM drugstbl
where uniqueid in (163740,206473);

SELECT uniqueid,upper(drugname) as drugname FROM drugstbl
where uniqueid in (206473);



In [0]:
%sql
desc history drugstbl;

In [0]:
%sql
desc history delta.`/Volumes/lakehousecat1/deltadb/datalake/targetdir`;

select * from delta.`/Volumes/lakehousecat1/deltadb/datalake/targetdir` version as of 4 where uniqueid=163740;

#####c. File DML (Update/Delete)
We don't do file DML usually, we are doing here just for learning about 
- file also can be undergone with limited DML operation
- we need to learn about how the background delta operation is happening when i do DML

In [0]:
# read delta files and create DF 
spark.read.format('delta').option("versionAsOf",4).load('/Volumes/lakehousecat1/deltadb/datalake/targetdir').where('uniqueid=163740').show()

In [0]:
#DML on Files: How to update delta files (Not used very frequently)
from delta.tables import DeltaTable
deltafile = DeltaTable.forPath(spark, "/Volumes/lakehousecat1/deltadb/datalake/targetdir")
deltafile.update("uniqueid=163740", { "rating": "rating - 1" } )


In [0]:
%sql

update delta.`/Volumes/lakehousecat1/deltadb/datalake/targetdir` 
set rating=rating+2
where uniqueid=163740;
    


In [0]:
%sql
SHOW TBLPROPERTIES delta.`/Volumes/lakehousecat1/deltadb/datalake/targetdir`;

ALTER TABLE delta.`/Volumes/lakehousecat1/deltadb/datalake/targetdir`
SET TBLPROPERTIES ('delta.checkpointInterval' = '10');

In [0]:
display(spark.read.parquet("/Volumes/lakehousecat1/deltadb/datalake/targetdir/_delta_log/00000000000000000020.checkpoint.parquet"))

In [0]:
%sql
select * from delta.`/Volumes/lakehousecat1/deltadb/datalake/targetdir` version as of 20 where uniqueid=163740 

In [0]:
%sql
create or replace table drugstbl_from_volume
as select * from delta.`/Volumes/lakehousecat1/deltadb/datalake/targetdir`;

-- select * from orc.``
-- select * from parquet.``

In [0]:
%sql
describe history delta.`/Volumes/lakehousecat1/deltadb/datalake/targetdir`;

reading a table simple - sql

reading delta from dir -  delta provoided function
                       - delta.`path`  we can able to perform sql quries 

In [0]:
spark.sql("select * from delta.`/Volumes/lakehousecat1/deltadb/datalake/targetdir` VERSION AS OF 12").show()

In [0]:
%fs
cp /Volumes/lakehousecat1/deltadb/datalake/targetdir/part-00000-9a3f0271-c494-49bd-9e8c-d2e6c962fcbd.c000.snappy.parquet  /Volumes/lakehousecat1/deltadb/datalake/

In [0]:
spark.read.parquet("/Volumes/lakehousecat1/deltadb/datalake/part-00000-9a3f0271-c494-49bd-9e8c-d2e6c962fcbd.c000.snappy.parquet").show()

In [0]:
spark.read.format('delta').load('/Volumes/lakehousecat1/deltadb/datalake/targetdir').where('uniqueid=163740').show()

#####d. File Delete

In [0]:
df=spark.read.format("delta").load('/Volumes/lakehousecat1/deltadb/datalake/targetdir')
df.where('uniqueid=206473').show()

In [0]:
from delta.tables import DeltaTable
deltaTable = DeltaTable.forPath(spark, "/Volumes/lakehousecat1/deltadb/datalake/targetdir")
deltaTable.delete("uniqueid=206473")

In [0]:
#Latest version data will be queried by default
df=spark.read.format("delta").load('/Volumes/lakehousecat1/deltadb/datalake/targetdir')
df.where('uniqueid=206473').show()

#####d. Merge Operation

In [0]:
%sql
select count(*) from drugstbl;

In [0]:
%sql
--CTAS (Create table As Select)
create or replace table drugstbl_merge as select * from drugstbl where rating<=8;

In [0]:
%sql
select 5700-2899

In [0]:
%sql
select count(*) from drugstbl_merge;
--5700
--2899

-- drugs - 5700 -> 2899 (merge)

-- 2899 update 
-- 2801 insert

select * from drugstbl_merge limit 10;

In [0]:
%sql
--merge syntax
--merge into targetable using sourcetable on condition_to_join
--when matched then update
--when not matched then insert
--when not matched by source then delete (some additional data in the target should be deleted, which is already deleted in source)
--Delta table support merge operation for (insert/update/delete)
--2899 updated
--2801 inserted
MERGE INTO drugstbl_merge tgt--2899
USING drugstbl src--5700
ON tgt.uniqueid = src.uniqueid
WHEN MATCHED THEN--2899 update
  UPDATE SET tgt.usefulcount= src.usefulcount,
             tgt.drugname = src.drugname,
             tgt.condition = src.condition
WHEN NOT MATCHED--2801 insert
  THEN INSERT (uniqueid,rating,date,usefulcount, drugname, condition ) VALUES (uniqueid,rating,date,usefulcount, drugname, condition);
  --5700-2899 = 2801

In [0]:
%sql
select count(*) from drugstbl_merge;

In [0]:
%sql
--What if the source got some data removed and which is present in the target still (we can leave it or delete)
insert into drugstbl_merge select 99999999,drugname,condition,rating,date,usefulcount 
from drugstbl limit 1;

In [0]:
%sql
--Target table contains excessive data
select count(*) from drugstbl_merge;

In [0]:
%sql
--Source table got few data deleted
select count(*) from drugstbl;

In [0]:
%sql
--Delta table support merge operation for (delete)
--1 deleted (which is not present in the source (source system deleted it already, hence target also has to delete))
MERGE INTO drugstbl_merge tgt
USING drugstbl src
ON tgt.uniqueid = src.uniqueid
WHEN MATCHED THEN
  UPDATE SET tgt.usefulcount= src.usefulcount,
             tgt.drugname = src.drugname,
             tgt.condition = src.condition
WHEN NOT MATCHED
  THEN INSERT (uniqueid,rating,date,usefulcount, drugname, condition ) VALUES (uniqueid,rating,date,usefulcount, drugname, condition)
WHEN NOT MATCHED BY SOURCE THEN DELETE;

In [0]:
%sql
select count(*) from drugstbl_merge;

In [0]:
#Few points to consider regarding merge...
#1. Merge can be only applied on tables in Databricks delta
#2. Merge operation using spark with (library delta.tables.DeltaTable) DSL (not by using SQL) - SQL is better to use
from delta.tables import DeltaTable
print(spark.read.table("drugstbl").count())
print(spark.read.table("drugstbl_merge").count())
tgt = DeltaTable.forName(spark, "drugstbl_merge")
src = spark.table("drugstbl")
(
    tgt.alias("tgt")
    .merge(
        src.alias("src"),
        "tgt.uniqueid = src.uniqueid"
    )
    .whenMatchedUpdate(set={
        "usefulcount": "src.usefulcount",
        "drugname": "src.drugname",
        "condition": "src.condition"
    })
    .whenNotMatchedInsert(values={
        "uniqueid": "src.uniqueid",
        "rating": "src.rating",
        "date": "src.date",
        "usefulcount": "src.usefulcount",
        "drugname": "src.drugname",
        "condition": "src.condition"
    })
    .whenNotMatchedBySourceDelete()
    .execute() )


In [0]:
print(spark.read.table("drugstbl").count())
print(spark.read.table("drugstbl_merge").count())

####3. Additional Operations on Deltalake & Deltatables

#####a. History & Versioning
*History* returns one row per commit/version and tells you what changed, when, and how.

In [0]:
%sql
DESC HISTORY drugstbl_merge

*Version as of* will reads the snapshot of drugstbl_merge at version 4 and Ignores all changes made in versions 5, 6, … current

In [0]:
%sql
--select * from (select * from deltadb.drugs version as of 2) where uniqueid=163740;
SELECT count(*) FROM drugstbl_merge  VERSION AS OF 0;--Behind the scene, databricks sql engine with the help of deltaengine (it will read the log and the respective data and produce the output)

#####b. Time Travel
*Timestamp as of* Reads the table as it existed at that exact timestamp and Any commits after the given timestamp is ignored

In [0]:
%sql
SELECT count(1) FROM drugstbl_merge TIMESTAMP AS OF '2026-08-10T09:58:07.000+00:00';

In [0]:
df=spark.range(1000)

df.repartition(4).write.format("delta").save("/Volumes/lakehousecat1/deltadb/datalake/range_delta")



In [0]:
%sql
desc history delta.`/Volumes/lakehousecat1/deltadb/datalake/range_delta`

#####c. Vaccum
*VACUUM* in Delta Lake removes old, unused files to free up storage, default retention hours is 168. These files come from operations like DELETE, UPDATE, or MERGE and are kept temporarily so time-travel queries can work.<br>

Before VACUUM<br>
Active + deleted parquet files exist<br>

After VACUUM<br>
Only ACTIVE parquet files remains and delete Old parquet files (from UPDATE/MERGE/DELETE)<br>
Logs remain intact<br>
Time travel beyond retention becomes impossible<br>

default 7 days 

table level we can set


vaccum tbl retail ? hours

In [0]:
%sql
--use lakehousecat1.deltadb;
DESC HISTORY prodcatalog.logistics.silver_staff;

In [0]:
%sql
select count(1) from prodcatalog.logistics.silver_staff TIMESTAMP AS OF '2026-01-23T03:24:19.000+00:00';

In [0]:
%sql
alter table drugstbl_merge SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = '15 hours');
VACUUM drugstbl_merge;--default value in databricks, we can reduce or increase this(but in serverless it is not possible to reduce)
SHOW TBLPROPERTIES drugstbl_merge



In [0]:
df=spark.read.table("drugstbl_merge").filter("uniqueid=159672")
#.filter(uniqueid=159672)
df.show(10)
df.printSchema()

In [0]:
%sql
use lakehousecat1.deltadb;
SET delta.retentionDurationCheck.enabled = false;
VACUUM drugstbl_merge retain 1 hours;

In [0]:
%sql
SELECT count(1) FROM drugstbl_merge TIMESTAMP AS OF '2026-01-23T03:24:19.000+00:00';

AI Suggested feature<br>
Setting the Delta table property 'delta.deletedFileRetentionDuration' to less than the default (1 week) is generally not recommended for production environments. Lowering the retention duration can lead to data loss if you need to time travel or restore data, as older files may be deleted sooner than expected. The default of 168 hours (1 week) is chosen to balance storage costs and safety for production workloads. Only reduce this value if you fully understand the risks and have a strong operational reason to do so

In [0]:
%sql

-- db -168 hr
--These properties can't be set in serverless, we will see this in cluster or i will get a table with more than 1 week data
--SET spark.databricks.delta.retentionDurationCheck.enabled = false;
--alter table drugstbl_merge SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = '24 hours');


In [0]:
%sql
alter table drugstbl_merge SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = '24 hours');

In [0]:
%sql
--use lakehousecat1.deltadb;
DESC HISTORY drugstbl_merge;

In [0]:
%sql
--default 168 hours (1 week), less than 1 week will not work in serverless for performance and session state reason
VACUUM drugstbl_merge RETAIN 1 HOURS ;

In [0]:
%sql
SELECT count(1) FROM drugstbl_merge TIMESTAMP AS OF '2026-01-26T16:55:40.000+00:00';

In [0]:
spark.sql("VACUUM drugstbl_merge RETAIN 168 HOURS")

In [0]:
%sql
DESC HISTORY drugstbl_merge

#####d. ACID Transactions
**Delta Lake supports ACID transactions under the hood via a transaction log.**
| ACID        | In Databricks         |
| ----------- | --------------------- |
| Atomicity   | Every transactions are Individual Transactions / All or nothing  |
| Consistency | Schema + constraints  |
| Isolation   | Using Version/Time/restore we can isolate transactions, we can't use TCL (commit/rollback) |
| Durability  | Every transaction Always hit the disk (durable), but can be controlled by Transaction log |

In [0]:
%sql
use lakehousecat1.deltadb

In [0]:
%sql
describe history acid_demo_txn 

In [0]:
%sql
select * from acid_demo_txn;

In [0]:
%sql
CREATE OR REPLACE TABLE acid_demo_txn (
  id INT,
  amount INT
) USING DELTA;


In [0]:
%sql
select * from acid_demo_txn;

In [0]:
%sql
--All or nothing (Atomic/Individual Transaction) - help us make a transaction complete or fail, hence no partial data
INSERT INTO acid_demo_txn VALUES--One atomic (all or nothing) transaction is inserting 3 rows
(1, 100),
(2, 200),
--(3, '300');
(3, 300);
--This will make the entire transaction failed (all or nothing)

In [0]:
%sql
select * from acid_demo_txn;

In [0]:
%sql
select * from acid_demo_txn where id=1;

In [0]:
%sql
--Atomicity (Individual transaction that doesn't affect the other)
UPDATE acid_demo_txn SET amount = amount + 100 WHERE id = 1;--individual/atomic
--The above statement is atomic, hence the below statement take amount as 200 and added 200 more
UPDATE acid_demo_txn SET amount = amount + 200 WHERE id = 1;--individual/atomic
describe history acid_demo_txn;
--START TRANSACTION; UPDATE acid_demo_txn SET amount = amount + 100 WHERE id = 1; commit;
--START TRANSACTION; UPDATE acid_demo_txn SET amount = amount + 200 WHERE id = 1; commit;

In [0]:
%sql
select * from acid_demo_txn where id=1;

In [0]:
%sql
--Apply constraint for maintaining consistancy
--We can apply in databricks deltatable, 2 types of constraints (check and not null), 
-- in other DBs we can use primary key, foreign key and unique constraints also..
ALTER TABLE acid_demo_txn ADD CONSTRAINT positive_amount CHECK (amount > 0);
INSERT INTO acid_demo_txn VALUES (4, 100);--Atomicity and consistancy
INSERT INTO acid_demo_txn VALUES (5, -100);--Atomicity and consistancy

In [0]:
%sql
--Only consistant data is loaded
select * from acid_demo_txn;

In [0]:
%sql
--Isolation (We can achieve using timetravel (restore operation (no rollback)))
--START TRANSACTION; SAVEPOINT before_delete; DELETE FROM employees WHERE employee_id = 129; ROLLBACK TO before_delete;
--Notebook1 (We can see the data in notebook1)
UPDATE acid_demo_txn SET amount = 999 WHERE id = 2;--This update will write the data in the disk with version added
--Notebook2 (We can see the data in notebook2 )
--use lakehousecat1.deltadb;
select * from (select * from acid_demo_txn version as of 14) where id=2;--serializable read (after the data successfully committed)

In [0]:
%sql
select * from acid_demo_txn;

In [0]:
%sql
--something like savepoint+rollback (but not really a rollback (TCL is not available in Databricks in the name of commit, rollback, savepoint))
restore table acid_demo_txn to version as of 13;

In [0]:
%sql
describe history acid_demo_txn;
--select * from acid_demo_txn;

In [0]:
%sql
--Durability (Despite of terminate and starting back the serverless, data still survives durably)
INSERT INTO acid_demo_txn VALUES (5, 500);

In [0]:
%sql
use lakehousecat1.deltadb;
select * from acid_demo_txn;

#####e. Transactions Control (TCL cannot be achieved using commit/rollback/savepoint)
Bigdata ecosystems such as spark/databricks/delta are not Transaction in nature, hence it will not support TCL directly, but can be achieved using version/timetravel/restore

In [0]:

%sql
--select count(1) from deltadb.drugs where date>'2012-02-28';
--4329
--Equivalent to delete and commit (with version (savepoint))
delete from drugstbl where date>'2012-02-28';

In [0]:
%sql 
select count(1) from drugstbl;

In [0]:
%sql
describe history drugstbl;

In [0]:
%sql
select * from acid_demo_txn;

In [0]:
%sql

describe history acid_demo_txn

In [0]:
%sql
--Equivalent to restore to a version
--Equivalent to Rollback to a savepoint
RESTORE TABLE acid_demo_txn TO VERSION AS OF 1;

In [0]:
%sql
select count(1) from drugstbl;

In [0]:
%sql
--We can restore to any older/later version (unlit 168 hours/vacumm period)
RESTORE TABLE drugstbl TO VERSION AS OF 4;

In [0]:
%sql
describe history drugstbl;

In [0]:
%sql
select count(1) from drugstbl;

## Drop and undrop

In [0]:
%sql

use lakehousecat1.deltadb ;

create or replace table emp_drop(eid int,ename string);

insert into emp_drop values(1,'a'),(2,'b'),(3,'c');

insert into emp_drop values(4,"raja");

select * from emp_drop;
    
describe history emp_drop;


drop table emp_drop;

In [0]:
%sql
show catalogs;
show schemas;
show tables in  lakehousecat1.deltadb;

show tables dropped in  lakehousecat1.deltadb;


undrop table emp_drop;

select * from emp_drop;

describe history emp_drop;  -- 7 days 



-- we took the backup of underlying dir (delta_log , all.parq)
-- emp  ---> emp.bkp 

-- create table cat.sche.emp using delta location 's3://buck/emp.bkp'; 


In [0]:
# create dataframe with emp id,name age
df = spark.createDataFrame([(1,'krish',20),(2,'raja',30),(3,'abc',40)],['id','name','age'])

df.display()
# create dataframe with dept id, dept name
df.write.format("delta").save("/Volumes/lakehousecat1/deltadb/datalake/dv_demo1")


In [0]:
%sql

describe history delta.`/Volumes/lakehousecat1/deltadb/datalake/dv_demo1`;


show tblproperties delta.`/Volumes/lakehousecat1/deltadb/datalake/dv_demo1`;

-- enable / disable the dv feature 
alter table delta.`/Volumes/lakehousecat1/deltadb/datalake/dv_demo1` 
set tblproperties (delta.enableDeletionVectors = false);


update delta.`/Volumes/lakehousecat1/deltadb/datalake/dv_demo1` set name = 'sachin' where id = 2;

delete from  delta.`/Volumes/lakehousecat1/deltadb/datalake/dv_demo1` where id = 2;
    
desc history  delta.`/Volumes/lakehousecat1/deltadb/datalake/dv_demo1`;



In [0]:
df = spark.createDataFrame([(1,'krish',20),(2,'raja',30),(3,'abc',40)],['id','name','age'])

df.display()
# create dataframe with dept id, dept name
df.write.format("delta").mode("overwrite").save("/Volumes/lakehousecat1/deltadb/datalake/dv_demo2")

In [0]:
%sql
describe history delta.`/Volumes/lakehousecat1/deltadb/datalake/dv_demo2`;


show tblproperties delta.`/Volumes/lakehousecat1/deltadb/datalake/dv_demo2`;


update delta.`/Volumes/lakehousecat1/deltadb/datalake/dv_demo2` set name = 'sachin' where id = 2;

delete from  delta.`/Volumes/lakehousecat1/deltadb/datalake/dv_demo2` where id = 2;

In [0]:
%sql

desc history delta.`/Volumes/lakehousecat1/deltadb/datalake/dv_demo2`;

In [0]:
%sql

vacuum delta.`/Volumes/lakehousecat1/deltadb/datalake/targetdir` dry run;
